# `bz-ggml`: Turnkey Dual Tesla T4 Inference on Kaggle

High-performance C++ inference engine for **Breeze-TTS-2** powered by **GGML** on Dual NVIDIA Tesla T4 GPUs.

### ⚡ Fast-Start Highlights:
- **Zero Wait Time**: Uses precompiled CUDA `sm_75` binaries (`breeze-cuda-sm75.tar.gz`) from GitHub. Startup takes **< 15 seconds** instead of compiling for 30 minutes!
- **Unlocked 2,040 Context**: Long-form speech generation up to ~2 minutes 10 seconds in a single pass.
- **Dual Architecture Choices**:
  1. **Option 1 (Headless Production Engine)**: Clean, zero-UI dual-GPU cluster inference for batch and microservices.
  2. **Option 2 (Gradio Studio UI)**: Interactive web studio with real-time 32-frame streaming playback, seekable master waveform, and voice persona management.

## 1. Hardware Verification
Confirm two NVIDIA Tesla T4 GPUs (16 GB VRAM each) are active.

In [ ]:
!nvidia-smi

## 2. Fast Turnkey Setup (< 15 Seconds Total)
Clones the repo and downloads precompiled CUDA  binaries from GitHub (bypassing the 30-minute compile).

In [ ]:
import os, subprocess, time

t0 = time.time()
print("🚀 [1/3] Symlinking CUDA driver...")
subprocess.run("ln -sf /usr/local/nvidia/lib64/libcuda.so /usr/lib/x86_64-linux-gnu/libcuda.so && ldconfig", shell=True)

# Clone repo
if not os.path.exists("/kaggle/working/bz-ggml"):
    print("📦 [2/3] Cloning bz-ggml engine...")
    subprocess.run("git clone -b feature/10-out-of-10-symmetric-architecture https://github.com/sharjeel103/bz-ggml.git /kaggle/working/bz-ggml", shell=True)

# Download precompiled CUDA sm_75 binaries directly from GitHub
os.makedirs("/kaggle/working/bz-ggml/build", exist_ok=True)
bin_tar = "/kaggle/working/bz-ggml/breeze-cuda-sm75.tar.gz"
breeze_so = "/kaggle/working/bz-ggml/build/libbreeze.so"
if not os.path.exists(breeze_so):
    print("⚡ [3/3] Downloading precompiled CUDA binaries from GitHub (~39 MB in 2 seconds)...
")
    BIN_URL = "https://github.com/sharjeel103/bz-ggml/raw/prebuilt-t4/breeze-cuda-sm75.tar.gz"
    subprocess.run(f"wget -q -c {BIN_URL} -O {bin_tar}", shell=True)
    subprocess.run(f"tar -xzf {bin_tar} -C /kaggle/working/bz-ggml/", shell=True)
    # Extract internal files into build/
    subprocess.run("cp -P /kaggle/working/bz-ggml/breeze-cuda-sm75/lib/* /kaggle/working/bz-ggml/build/", shell=True)

# Download official Q8_0 model (~3.4 GB)
os.makedirs("/kaggle/working/models", exist_ok=True)
model_path = "/kaggle/working/models/breeze-tts-2-q8_0.gguf"
if not os.path.exists(model_path):
    print("📥 Downloading breeze-tts-2-q8_0.gguf from Hugging Face (~15 seconds)...
")
    from huggingface_hub import hf_hub_download
    hf_hub_download(repo_id="HoppouAI/Breeze-TTS-2.cpp", filename="breeze-tts-2-q8_0.gguf", local_dir="/kaggle/working/models")

print(f"
✅ Turnkey setup ready in {time.time()-t0:.1f}s!")

---
## 🌟 Code Type 1: Headless Production Server / Engine (Zero Gradio)
Run pure dual-GPU cluster inference with zero UI dependencies.

In [ ]:
import sys, os, time
sys.path.insert(0, "/kaggle/working/bz-ggml/python")
from bz_ggml import DualInstanceCluster, UserTask
from IPython.display import Audio, display

# 1. Initialize Dual-GPU Cluster across GPU 0 and GPU 1
cluster = DualInstanceCluster(
    model_path="/kaggle/working/models/breeze-tts-2-q8_0.gguf",
    lib_path="/kaggle/working/bz-ggml/build/libbreeze.so",
    enable_vocoder=True,
    vocoder_mode="on"
)

# 2. Submit synthesis task
task = UserTask(
    id=101,
    text="The production inference engine is fully operational on dual Tesla T4 GPUs with zero Gradio overhead.",
    instruction="Speak with natural clarity and professional authority.",
    cfg_scale=1.0,
    seed=42
)

print("🎙️ Synthesizing speech via headless production engine...")
results = cluster.run_workload([task], out_dir="/kaggle/working/server_audio")
res = results[0]

print(f"
🎉 Generation Completed!")
print(f"  • Worker:      {res.worker}")
print(f"  • Audio Time:  {res.audio_s:.2f}s ({res.words} words)")
print(f"  • TTFA:        {res.ttfa_s * 1000:.1f} ms")
print(f"  • RTF:         {res.rtf:.3f} (< 1.0 = faster than real-time)")
print(f"  • Output WAV:  {res.wav_path}")

display(Audio(res.wav_path))

---
## 🌟 Code Type 2: Interactive Gradio Studio (Web UI + Public Link)
Launch the full studio UI on port 7860 with live streaming, voice cloning, and public tunneling.

In [ ]:
import subprocess, time, os

# Install optional whisper ASR for voice registration
subprocess.run("pip install -q faster-whisper gradio", shell=True)
os.makedirs("/kaggle/working/data/voices", exist_ok=True)

# Kill any existing processes
subprocess.run("pkill -9 -f gradio_app.py; pkill -9 -f frpc", shell=True)
time.sleep(2)

env_vars = (
    "export PYTHONPATH=/kaggle/working/bz-ggml/python: && "
    "export PYTHONUNBUFFERED=1 && "
)

cmd = (
    env_vars +
    "nohup python3 -u /kaggle/working/bz-ggml/python/gradio_app.py "
    "--model /kaggle/working/models/breeze-tts-2-q8_0.gguf "
    "--lib /kaggle/working/bz-ggml/build/libbreeze.so "
    "--vocoder-mode on "
    "--voices-dir /kaggle/working/data/voices "
    "--asr-device 1 "
    "--share --port 7860 > /kaggle/working/gradio_q8.log 2>&1 &"
)

subprocess.run(cmd, shell=True, executable="/bin/bash")
print("🚀 Gradio Studio launched on port 7860!")
print("⏳ Waiting for public Gradio tunnel URL...
")

public_url = None
for _ in range(60):
    time.sleep(2)
    if os.path.exists("/kaggle/working/gradio_q8.log"):
        with open("/kaggle/working/gradio_q8.log", "r") as f:
            for line in f.read().splitlines():
                if "Running on public URL:" in line:
                    public_url = line.split("Running on public URL:")[1].strip()
                    break
    if public_url:
        break

print("=" * 65)
print(f"  🌟 PUBLIC GRADIO LINK:  {public_url}")
print("=" * 65)